In [1]:
# Install required libraries just in case Colab doesn't have them updated
!pip install -q transformers torch pillow

#### **1. INITIALIZATION**

In [2]:
import os
import torch
import urllib.request
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

#### **2. DOWNLOAD CLIP MODEL TO DRIVE FOLDER**

In [3]:
# 1. Define where to save the model on your Drive
save_path = "/content/drive/MyDrive/Model/clip-vit-large-patch14"

# print("Step 1: Downloading CLIP model from Hugging Face...")
# # We use the standard base model.Can upgrade to "openai/clip-vit-large-patch14" later if needed.
# model_id = "openai/clip-vit-large-patch14"

processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id)

print(f"Step 2: Saving model permanently to {save_path}...")
processor.save_pretrained(save_path)
model.save_pretrained(save_path)
print("Save complete!\n")

Step 1: Downloading CLIP model from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step 2: Saving model permanently to /content/drive/MyDrive/Model/clip-vit-large-patch14...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Save complete!



#### **3. TESTING MODEL**

In [7]:
save_path = "/content/drive/MyDrive/Model/clip-vit-large-patch14"
# ---------------------------------------------------------
print("Step 3: Reloading model from your LOCAL Drive folder to verify...")
local_processor = CLIPProcessor.from_pretrained(save_path)
local_model = CLIPModel.from_pretrained(save_path)

print("Step 4: Running a quick test...")
# Let's download a quick sample image of a cat from the internet to test it
urllib.request.urlretrieve("https://cdn.britannica.com/06/177306-050-C2A56017/David-Beckham.jpg", "test_image.jpg")
image = Image.open("test_image.jpg")

# The text labels we want CLIP to classify
texts = ["Generated Image", "Real Image"]

# Process the image and text together
inputs = local_processor(text=texts, images=image, return_tensors="pt", padding=True)

# Run it through the model
with torch.no_grad():
    outputs = local_model(**inputs)
    logits_per_image = outputs.logits_per_image
    # Convert the raw scores into percentages using Softmax
    probs = logits_per_image.softmax(dim=1)

print("\n --- TEST RESULTS ---")
for text, prob in zip(texts, probs[0]):
    print(f"Probability of '{text}': {prob.item() * 100:.2f}%")

Step 3: Reloading model from your LOCAL Drive folder to verify...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Step 4: Running a quick test...

 --- TEST RESULTS ---
Probability of 'AI generated image': 68.38%
Probability of 'real image': 31.62%
